### Gold Layer

In [0]:
from pyspark.sql.functions import *

In [0]:
# Medallion architecture ----
# Load data from silver layer , perform transformation ,
# create fact and dimention table and analysis the business needs then write the data into gold layer

In [0]:
# Customers Table
customers_df = spark.table('workspace.silver_layer.customers')

# Order_items Table
order_items_df = spark.table('workspace.silver_layer.order_items')

# Orders Table
orders_df = spark.table('workspace.silver_layer.orders')

# Products Table
products_df = spark.table('workspace.silver_layer.products')

# Sellers table
sellers_df = spark.table('workspace.silver_layer.sellers')

### Create Dimension and Fact Tables

In [0]:
# Customer Dimension 
dim_customers = customers_df.select(
    col("customer_id"),
    col("customer_city"),
    col("customer_state"))

In [0]:
# Product Dimension
dim_products = products_df.select(
    col("product_id"),
    col("product_category_name"))

In [0]:
# Seller Dimension
dim_sellers = sellers_df.select(
    col("seller_id"),
    col("seller_city"),
    col("seller_state"))

In [0]:
# Orders Fact
fact_orders = (
    order_items_df.alias("oi")
    .join(orders_df.alias("o"), col("oi.order_id") == col("o.order_id"), "inner")
    .select(
        col("oi.order_id").alias("order_id"),   
        col("o.customer_id"),                  
        col("oi.product_id"),
        col("oi.seller_id"),
        col("o.order_purchase_timestamp").alias("order_date"),
        col("oi.price"),
        col("oi.freight_value"),
        col("oi.total_amount")))

In [0]:
# Display fact table
fact_orders.display()

#### Spark SQL

In [0]:
# Create sql view
dim_customers.createTempView("dim_customers_view")
dim_products.createTempView("dim_products_view")
dim_sellers.createTempView("dim_sellers_view")
fact_orders.createTempView("fact_orders_view")

In [0]:
%sql
SELECT * FROM dim_customers_view LIMIT 10;

In [0]:
# Revenue by State

In [0]:
%sql
SELECT
c.customer_state,
ROUND(SUM(o.total_amount), 2) AS total_revenue
FROM fact_orders_view o
JOIN dim_customers_view c
ON o.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY total_revenue DESC

In [0]:
# Top Selling Product Categories

In [0]:
%sql
SELECT
p.product_category_name,
round(SUM(f.total_amount),2) AS revenue
FROM fact_orders_view f
JOIN dim_products_view p
ON f.product_id = p.product_id
GROUP BY p.product_category_name
ORDER BY revenue DESC
LIMIT 10

In [0]:
## Monthly Sales Trend

In [0]:
%sql
SELECT
month(order_date) AS month,
round(SUM(total_amount),2) AS monthly_sales
FROM fact_orders_view
GROUP BY month
ORDER BY month

#### Write data into gold layer

In [0]:
dim_customers.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("workspace.gold_layer.dim_customers")

dim_products.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("workspace.gold_layer.dim_products")

dim_sellers.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("workspace.gold_layer.dim_sellers")

fact_orders.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("workspace.gold_layer.fact_orders")